<a href="https://colab.research.google.com/github/Yennanng/PTDLNC/blob/develop/PTDLNC_B%C3%80I_NH%C3%93M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**BÀI 2: Bài tập Classification: Dự đoán khách hàng rời bỏ ngân hàng (Churn Prediction)**

**Bước 1: Khám phá dữ liệu (EDA)**

In [ ]:
#load data
import gdown
url = "https://drive.google.com/uc?id=12dOGAhw3nZ104_OQsURmvW2M0Gj2iyM4"
data_filename = "churn.csv"
gdown.download(url, data_filename, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=12dOGAhw3nZ104_OQsURmvW2M0Gj2iyM4
To: /content/churn.csv
100%|██████████| 685k/685k [00:00<00:00, 30.2MB/s]


'churn.csv'

In [ ]:
# Đọc dữ liệu
import pandas as pd
df = pd.read_csv("churn.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# Mô tả số dòng, số cột
print("Kích thước dữ liệu (Dòng, Cột):", df.shape)

Kích thước dữ liệu (Dòng, Cột): (10000, 14)


In [ ]:
#Tính tỷ lệ khách hàng rời đi (Exited=1) và ở lại (Exited=0)
churn_counts = df['Exited'].value_counts()
churn_percentage = df['Exited'].value_counts(normalize=True) * 100
print("\nSố lượng khách hàng theo nhóm:")
print(churn_counts)
print("\nTỷ lệ phần trăm (%):")
print(churn_percentage)


Số lượng khách hàng theo nhóm:
Exited
0    7963
1    2037
Name: count, dtype: int64

Tỷ lệ phần trăm (%):
Exited
0    79.63
1    20.37
Name: proportion, dtype: float64


In [ ]:
# Bước 2. Tiền xử lý dữ liệu
# ---------------------------------------------------------

# 2.1. # Loại bỏ cột RowNumber, CustomerId, Surname vì không có giá trị dự báo
df_processed = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df_processed.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Loại bỏ cột định danh: RowNumber, CustomerId, Surname.

df_processed = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# 1. Label Encoding cho Gender: Chuyển Female -> 0, Male -> 1
le = LabelEncoder()
df_processed['Gender'] = le.fit_transform(df_processed['Gender'])

# 2. One-Hot Encoding cho Geography: Chuyển sang 0 và 1 (dtype=int)
df_processed = pd.get_dummies(df_processed, columns=['Geography'], dtype=int)

# Hiển thị kết quả
df_processed.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


In [ ]:
#Chuẩn hoá các biến số dùng StandardScaler

from sklearn.preprocessing import StandardScaler

# 1. Danh sách các biến số cần chuẩn hoá
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

# 2. Khởi tạo công cụ StandardScaler
scaler = StandardScaler()

# 3. Tạo một bản sao dữ liệu mới để chuẩn hoá
# (Giữ lại df_processed gốc cho các mô hình dựa trên cây như Decision Tree/Random Forest)
df_logistic = df_processed.copy()

# 4. Thực hiện chuẩn hoá các cột số
df_logistic[num_cols] = scaler.fit_transform(df_logistic[num_cols])

# Hiển thị kết quả sau khi chuẩn hoá
df_logistic.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,-0.326221,0,0.293517,-1.041760,-1.225848,-0.911583,1,1,0.021886,1,1,0,0
1,-0.440036,0,0.198164,-1.387538,0.117350,-0.911583,0,1,0.216534,0,0,0,1
2,-1.536794,0,0.293517,1.032908,1.333053,2.527057,1,0,0.240687,1,1,0,0
3,0.501521,0,0.007457,-1.387538,-1.225848,0.807737,0,0,-0.108918,0,1,0,0
4,2.063884,0,0.388871,-1.041760,0.785728,-0.911583,1,1,-0.365276,0,0,0,1


In [ ]:
# Chia dữ liệu thành train/test (80/20).
from sklearn.model_selection import train_test_split

# 1. Tách biến mục tiêu (y) - chung cho cả hai mô hình
y = df_processed['Exited']

# 2. Chia dữ liệu cho các mô hình dựa trên Cây (Decision Tree, Random Forest)
# Sử dụng df_processed (dữ liệu chưa chuẩn hóa)
X_tree = df_processed.drop('Exited', axis=1)
X_train_tree, X_test_tree, y_train, y_test = train_test_split(X_tree, y, test_size=0.2, random_state=42, stratify=y)

# 3. Chia dữ liệu cho mô hình Logistic Regression
# Sử dụng df_logistic (dữ liệu đã chuẩn hóa StandardScaler)
X_log = df_logistic.drop('Exited', axis=1)
X_train_log, X_test_log, _, _ = train_test_split(X_log, y, test_size=0.2, random_state=42, stratify=y)

# --- Kiểm tra kích thước và tỷ lệ ---
print("--- KÍCH THƯỚC DỮ LIỆU ---")
print(f"Kích thước tập Train: {X_train_tree.shape}")
print(f"Kích thước tập Test:  {X_test_tree.shape}")

print("\n--- TỶ LỆ KHÁCH HÀNG RỜI BỎ (EXITED = 1) ---")
train_rate = y_train.value_counts(normalize=True)[1] * 100
test_rate = y_test.value_counts(normalize=True)[1] * 100

print(f"Tỷ lệ trong tập Train: {train_rate:.2f}%")
print(f"Tỷ lệ trong tập Test:  {test_rate:.2f}%")

--- KÍCH THƯỚC DỮ LIỆU ---
Kích thước tập Train: (8000, 12)
Kích thước tập Test:  (2000, 12)

--- TỶ LỆ KHÁCH HÀNG RỜI BỎ (EXITED = 1) ---
Tỷ lệ trong tập Train: 20.38%
Tỷ lệ trong tập Test:  20.35%
